In [ ]:
import sys
sys.path.append('..')

import os
import pandas as pd
import time
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import RAG pipeline
from src.rag_pipeline import RAGPipeline

import os
from dotenv import load_dotenv
load_dotenv() 

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [2]:
test_questions = [
    "Why are customers unhappy with credit card fees?",
    "What are the main issues with money transfers?",
    "How do customers complain about unauthorized charges?",
    "What are the top complaint issues overall?",
    "How do customers describe fraud or scam experiences?",
    "Which companies receive the most complaints?"
]

### TEST CUSTOM VECTOR STORE

In [3]:
def test_vector_store(rag, store_name):
    """Test a vector store with all questions."""
    print("\n" + "="*60)
    print(f"TESTING: {store_name}")
    print("="*60)
    
    results = []
    
    for i, question in enumerate(test_questions, 1):
        print(f"\n{'─'*60}")
        print(f"Q{i}: {question}")
        print('─'*60)
        
        start = time.time()
        result = rag.answer(question)
        elapsed = time.time() - start
        
        results.append({
            'question': question,
            'answer': result['answer'],
            'sources': result['sources'],
            'num_sources': result['num_sources'],
            'time': elapsed
        })
        
        print(f"\nANSWER:")
        print(result['answer'])
        print(f"\nSOURCES: {result['num_sources']}")
        print(f"TIME: {elapsed:.2f}s")
    
    return results

In [4]:
print("\n" + "="*60)
print("CUSTOM VECTOR STORE")
print("="*60)

# Initialize custom vector store
print("\nLoading custom vector store...")
rag_custom = RAGPipeline(
    vector_store_path='../vector_store',
    use_prebuilt=False,
    use_llm=True,
    use_api=True,
    api_model="Qwen/Qwen2.5-7B-Instruct"
)

custom_results = test_vector_store(rag_custom, "CUSTOM VECTOR STORE")


CUSTOM VECTOR STORE

Loading custom vector store...
INITIALIZING RAG PIPELINE

 Loading custom vector store from: ../vector_store
   Loaded FAISS index: 36,512 vectors
   Loaded 36,512 chunks
   Loaded 36,512 metadata records
   Vector store loaded successfully!

 Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3552.48it/s]


   Model loaded
   Embedding dimension: 384

 Loading LLM via API: Qwen/Qwen2.5-7B-Instruct
   Using authenticated access
   API client ready: Qwen/Qwen2.5-7B-Instruct
------------------------------------------------------------

TESTING: CUSTOM VECTOR STORE

────────────────────────────────────────────────────────────
Q1: Why are customers unhappy with credit card fees?
────────────────────────────────────────────────────────────

ANSWER:
Customers are unhappy with credit card fees due to several reasons mentioned in the context:

1. Excessive fees: Customers feel that the fees being charged are excessive and not justified.
2. Lack of mercy from banks: Banks are perceived to be adding fees without consideration for the customers' financial situations.
3. Poor customer service: The level of customer service is described as mediocre or even rude and unprofessional, with some customers receiving negative comments from bank representatives.
4. Financial burden: The high fees are causing a

In [5]:
# Run tests with answers displayed
custom_results = []

print("\nRunning tests...")
for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*60}")
    print(f"Q{i}: {question}")
    print('='*60)
    
    start_time = time.time()
    result = rag_custom.answer(question)
    elapsed = time.time() - start_time
    
    custom_results.append({
        'question': question,
        'answer': result['answer'],
        'sources': result['num_sources'],
        'time': elapsed,
        'success': True
    })
    
    # Display the answer
    print(f"\n  ANSWER:")
    print(result['answer'])
    
    print(f"\nSOURCES: {result['num_sources']}")
    print(f"⏱ TIME: {elapsed:.2f}s")


Running tests...

Q1: Why are customers unhappy with credit card fees?

  ANSWER:
Customers are unhappy with credit card fees due to several reasons mentioned in the context:

1. Excessive fees: Customers feel that the fees being charged are excessive, as stated in Excerpt 2.
2. Lack of mercy from banks: Customers mention that banks keep adding fees to their credit card balances without any consideration for their financial situation, as described in Excerpt 1.
3. Poor customer service: The level of customer service is also a point of dissatisfaction, with some customers reporting rudeness and unprofessionalism from customer service representatives, as noted in Excerpt 2.
4. Financial burden: The high fees are causing a significant financial strain on customers, forcing them to use savings to pay off their credit card balances, as mentioned in Excerpt 1.
5. Unauthorized transactions and payment difficulties: Some customers are facing issues with unauthorized transactions and are incur

### TEST PRE-BUILT VECTOR STORE

In [6]:
prebuilt_path = '../data/processed/complaint_embeddings.parquet'

if os.path.exists(prebuilt_path):
    print(f"\nLoading pre-built vector store...")
    
    rag_prebuilt = RAGPipeline(
        vector_store_path=prebuilt_path,
        use_prebuilt=True,
        use_llm=True,
        use_api=True,
        api_model="Qwen/Qwen2.5-7B-Instruct"
    )
    
    print(f"\n Pre-built vector store loaded:")
    print(f"   Chunks: {len(rag_prebuilt.chunks):,}")
    print(f"   Vectors: {rag_prebuilt.index.ntotal:,}")
    
    prebuilt_results = test_vector_store(rag_prebuilt, "PRE-BUILT VECTOR STORE")

    use_prebuilt = True
else:
    print(f"\n  Pre-built not found at: {prebuilt_path}")
    print("   Only custom vector store results available.")
    use_prebuilt = False
    prebuilt_results = []


Loading pre-built vector store...
INITIALIZING RAG PIPELINE

 Loading pre-built vector store from: ../data/processed/complaint_embeddings.parquet
   Loaded 1,375,327 records
   Columns: ['id', 'document', 'embedding', 'metadata']
   Memory: 1076.19 MB

   Extracting chunks and metadata...
   Chunks: 1,375,327
   Metadata columns: ['id', 'metadata']

   Creating FAISS index...
   Embedding dimension: 384
   Index size: 1,375,327 vectors
   Index memory: 4029.28 MB
   Vector store loaded successfully!

 Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1105.93it/s]


   Model loaded
   Embedding dimension: 384

 Loading LLM via API: Qwen/Qwen2.5-7B-Instruct
   Using authenticated access
   API client ready: Qwen/Qwen2.5-7B-Instruct
------------------------------------------------------------

 Pre-built vector store loaded:
   Chunks: 1,375,327
   Vectors: 1,375,327

TESTING: PRE-BUILT VECTOR STORE

────────────────────────────────────────────────────────────
Q1: Why are customers unhappy with credit card fees?
────────────────────────────────────────────────────────────

ANSWER:
Customers are unhappy with credit card fees because the fees and interest rates increase significantly once a balance is carried, making it difficult to pay down the debt. Additionally, customers feel that credit card companies are not transparent and do not provide timely information, often sending notices of fees long after the issue has occurred. There is also a perception that credit card companies are targeting less fortunate consumers who may struggle to manage these

### QUALITY SCORING 

In [14]:
def score_answer(question, answer, sources, num_sources=None):
    """
    Score answer quality on 1-5 scale.
    Handles both custom and pre-built result structures.
    """
    score = 1
    
    if not answer or len(answer) < 20:
        return 1
    
    # Get number of sources (handles both int and list)
    if isinstance(sources, list):
        source_count = len(sources)
    elif isinstance(sources, int):
        source_count = sources
    elif num_sources is not None:
        source_count = num_sources
    else:
        source_count = 0
    
    if len(answer) > 50:
        score += 1
    if source_count >= 3:
        score += 1
    if any(word in answer.lower() for word in ['because', 'indicate', 'suggest', 'show', 'trend']):
        score += 1
    if len(answer) > 100 and answer[-1] in '.!?':
        score += 1
    if '1.' in answer or '•' in answer or ' - ' in answer:
        score += 1
    
    return min(score, 5)

# Score custom results
for r in custom_results:
    r['quality_score'] = score_answer(
        r['question'], 
        r['answer'], 
        r['sources']  # This is an int (5)
    )

# Score prebuilt results
if use_prebuilt:
    for r in prebuilt_results:
        r['quality_score'] = score_answer(
            r['question'], 
            r['answer'], 
            r['sources'],  # This is a list of dicts
            r['num_sources']  # This is the count (5)
        )


# Verify
print(f"Custom scores: {[r['quality_score'] for r in custom_results]}")
if use_prebuilt:
    print(f"Pre-built scores: {[r['quality_score'] for r in prebuilt_results]}")

Custom scores: [5, 5, 5, 5, 5, 5]
Pre-built scores: [5, 5, 4, 4, 4, 4]


### COMPARISON TABLE 

In [15]:
# ============================================================================
# STEP 6: COMPARISON TABLE
# ============================================================================

print("\n" + "="*60)
print("COMPARISON TABLE")
print("="*60)

comparison_data = []

for i, q in enumerate(test_questions):
    c = custom_results[i]
    
    # Get custom source count
    c_sources = c['sources'] if isinstance(c['sources'], int) else len(c['sources'])
    
    row = {
        'Question': q[:40] + '...' if len(q) > 40 else q,
        'Custom_Score': f"{c['quality_score']}/5",
        'Custom_Sources': c_sources,
        'Custom_Time': f"{c['time']:.2f}s",
    }
    
    if use_prebuilt:
        p = prebuilt_results[i]
        
        # Get pre-built source count
        p_sources = p['num_sources'] if 'num_sources' in p else len(p['sources'])
        
        row['Prebuilt_Score'] = f"{p['quality_score']}/5"
        row['Prebuilt_Sources'] = p_sources
        row['Prebuilt_Time'] = f"{p['time']:.2f}s"
        
        # Determine which performed better
        if c['quality_score'] > p['quality_score']:
            row['Better'] = 'Custom '
        elif p['quality_score'] > c['quality_score']:
            row['Better'] = 'Pre-built '
        else:
            row['Better'] = 'Tie'
    
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)


COMPARISON TABLE


,Question,Custom_Score,Custom_Sources,Custom_Time,Prebuilt_Score,Prebuilt_Sources,Prebuilt_Time,Better
0,Why are customers unhappy with credit ca...,5/5,5,2.58s,5/5,5,3.31s,Tie
1,What are the main issues with money tran...,5/5,5,1.84s,5/5,5,1.73s,Tie
2,How do customers complain about unauthor...,5/5,5,3.17s,4/5,5,1.96s,Custom
3,What are the top complaint issues overal...,5/5,5,2.97s,4/5,5,1.32s,Custom
4,How do customers describe fraud or scam ...,5/5,5,1.44s,4/5,5,1.54s,Custom
5,Which companies receive the most complai...,5/5,5,1.35s,4/5,5,1.43s,Custom
